# PolicyProof AI — CuratorKIT data preparation

**Lexsi Track 1 | Stage 2 | No GPU training.** This notebook loads only the original training split and uses the official CuratorKIT API to clean, deduplicate and export training data. It never reads the 48-case development holdout for training or runs GPU fine-tuning.

**Run in a new Colab notebook.** CPU runtime is sufficient. Your saved baseline JSON and review CSV are already in ChatGPT, so no need to keep a GPU session running for curation.

**Important:** The original 78 examples are synthetic and not yet human-reviewed. Curation is a useful integration test, not proof that training data quality is sufficient for a final competition result.

## 1 — Retrieve versioned project code and verify split integrity

In [ ]:
import os, pathlib, subprocess, sys, json
ROOT = pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
      "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git", str(ROOT)], check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,"-m","scripts.prepare"],check=True)
subprocess.run([sys.executable,"-m","scripts.audit_dataset","--data-dir","data"],check=True)
REV = subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Project revision:", REV)
print("Training examples only:",sum(1 for _ in open(ROOT/"data/train_alpaca.jsonl",encoding="utf-8")))

## 2 — Install CuratorKIT core, not the heavyweight extras

The current CuratorKIT quickstart supports local JSONL with the core installation and requires no LLM provider credentials. A fresh runtime is recommended to avoid changing a currently loaded Tiny Aya environment.

In [ ]:
%pip -q install "curatorkit>=0.1.15,<1"
import importlib.metadata as im
print("CuratorKIT version:", im.version("curatorkit"))

## 3 — Official CuratorKIT pipeline: cleaning, exact deduplication and export

This explicitly reads **only** `train_alpaca.jsonl`. No test or held-out examples are included.

In [ ]:
from curatorkit import Curator, CuratorConfig
from policyproof.core import sha256
input_path = ROOT / "data/train_alpaca.jsonl"
output_path = ROOT / "artifacts/curator_train_v1"
result = Curator(CuratorConfig(
    dataset=str(input_path),
    clean=True,
    dedup="exact",
    export_formats=["alpaca", "sharegpt"],
    output_dir=str(output_path),
)).run()
result.print_summary()
manifest_path = output_path / "manifest.json"
assert manifest_path.exists(), "CuratorKIT did not write expected provenance manifest"
print("Input SHA256:",sha256(input_path))
print("Manifest:",manifest_path)
print("Curated files:",[p.name for p in output_path.iterdir() if p.is_file()])

## 4 — Inspect curation output and guard against empty datasets

Review the manifest and rejection reasons. Data may be filtered; zero accepted rows must **stop** training rather than silently using the original file.

In [ ]:
from policyproof.core import read_jsonl
curated_path = output_path / "sft_alpaca.jsonl"
assert curated_path.exists(), "No Alpaca-format curation output. Inspect manifest and file names."
curated_rows = read_jsonl(curated_path)
assert curated_rows, "CuratorKIT kept no training examples. Stop."
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Curated rows:",len(curated_rows))
print("Source rows: 78 | All source examples fictional")
print("Curated SHA256:",sha256(curated_path))
print("Manifest keys:",list(manifest.keys()))
print("Sample schema:",sorted(curated_rows[0].keys()))
print("Example instruction excerpt:",str(curated_rows[0].get("instruction",""))[:250])
print("First rejection excerpts:")
reject = output_path/"rejected.jsonl"
if reject.exists():
    print(reject.read_text(encoding="utf-8")[:650] or "(no rejected rows)")

## 5 — Verify exact training SHA and safe archive structure before downloading

The notebook now rejects incomplete, altered, unsafe or unexpectedly large CuratorKIT archives locally, before giving you a ZIP to use for training. This check does not train a model.


In [ ]:
import shutil
from scripts.check_align_config import check_train_data
from scripts.restore_colab_session import EXPECTED_SHA
from scripts.verify_curator_archive import check_zip

# Fail before downloading any export that is incomplete or different from
# the original 78-example synthetic training-only artifact.
checked_train = check_train_data(output_path / "sft_alpaca.jsonl")
assert checked_train["row_count"] == 78
if checked_train["dataset_sha256"] != EXPECTED_SHA:
    raise RuntimeError(
        "CuratorKIT output has changed from the pinned synthetic training dataset. "
        "Inspect the exported source rows and tooling version before continuing."
    )

archive = shutil.make_archive(
    "/content/policyproof_curatorkit_train_v1",
    "zip", root_dir=output_path,
)
archive_report = check_zip(archive)
assert archive_report["status"] == "pass"
assert archive_report["row_count"] == 78
print("Verified source SHA:", checked_train["dataset_sha256"])
print("Bounded ZIP inventory:", archive_report["bounded_zip_inventory"])
print("CuratorKIT archive:", archive)

from google.colab import files
files.download(archive)

## What happens next?

Send the ZIP or a screenshot of the CuratorKIT manifest back to ChatGPT. We will check accepted rows, train-data integrity and framework compatibility before running any AlignTune QLoRA job. **Do not fit the model on the 48 already-inspected held-out questions**; create a fresh final holdout for credible claims.

Sources: [CuratorKIT docs](https://curatorkit.lexsi.ai/latest/getting-started/quickstart/) · [AlignTune docs](https://aligntune.lexsi.ai/user-guide/sft/) · [Project repo](https://github.com/betterlifestyle04-commits/PolicyProof-AI).